# Scenario comparison

Compares month-end **nominal** net worth across any number of scenarios against a
baseline, and finds each variant's break-even month — the point after which it
stays ahead for good.

Built for Social Security claim-timing questions: claiming later means a smaller
benefit stream starts later but at a higher monthly rate, so the variant trails
early and (usually) catches up later. The break-even date is where it does. The
same machinery answers "what if benefits are cut?" — add a scenario carrying the
cut and read the gap.

List the scenarios in the next cell, baseline first. Each must already have a run
(`analysis sim load <name> && analysis sim run <name>`); the newest run for each
is used.

> **Note:** after running cells, `git status` will show this notebook as modified
> even with no real changes — see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it).


In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from analysis.sim.connection import RunConnection
from analysis.sim.loader import resolve_scenario_name

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = Path(os.environ.get("ANALYSIS_DATA_DIR", REPO / "data" / "shared" / "sim"))

SCENARIOS = ["current", "january", "current-cut", "january-cut"]
BASELINE = SCENARIOS[0]
VARIANTS = [name for name in SCENARIOS if name != BASELINE]
COLORS = dict(zip(SCENARIOS, plt.rcParams["axes.prop_cycle"].by_key()["color"]))

refs, runs = {}, {}
for name in SCENARIOS:
    ref = resolve_scenario_name(DATA_DIR, name)
    if ref.run_path is None:
        raise FileNotFoundError(
            f"no run for {name!r}; run: analysis sim load {name} "
            f"&& analysis sim run {name}"
        )
    refs[name], runs[name] = ref, RunConnection(ref.run_path)

display(Markdown(f"Baseline: **`{BASELINE}`** — everything else is measured against it."))
display(pd.DataFrame(
    [{"scenario": name, "run": refs[name].run_path.name} for name in SCENARIOS]
).set_index("scenario"))


## Are these two runs comparable?

A comparison is only meaningful if both runs came from the same code and the same
data version — otherwise a difference here could be an unrelated edit rather than
the scenario change under test.


In [ ]:
_META = [
    "scenario_name", "status", "finished_at",
    "code_version", "code_dirty", "data_version", "data_dirty",
]
meta = pd.concat(
    [runs[name].items("simulation_run")[_META] for name in SCENARIOS],
    ignore_index=True,
)
for column in ("code_version", "data_version"):
    meta[column] = meta[column].str[:12]
display(meta.set_index("scenario_name").T)

_warnings = []
if meta["data_version"].nunique() > 1:
    _warnings.append("different **data versions** — the data repo changed between runs")
if meta["code_version"].nunique() > 1:
    _warnings.append("different **code versions** — the engine changed between runs")
if meta[["code_dirty", "data_dirty"]].to_numpy().any():
    _warnings.append("at least one run was made from **uncommitted** code or data")
if (meta["status"] != "succeeded").any():
    _warnings.append("at least one run did **not succeed**")
display(Markdown(
    "\n".join(f"> ⚠️ {w}" for w in _warnings)
    if _warnings else "> ✅ Same code and data version; differences below are the scenarios."
))


## What actually differs?

Each scenario inherits everything from its parent except the rows it overrides, so
this is the whole mechanism behind any difference in the charts below. If a rule
you expected to change isn't listed, the override didn't take effect.

Two sources of difference are shown separately:

- **Rules** — as they stand at the *end* of the run. A calendar event that edits a
  rule mid-run (`set_amount`) shows up here as a changed amount, because the run
  snapshot records the post-event state.
- **Calendar events** — one-time interventions, including the benefit cut.


In [ ]:
_COMPARE = ["amount_cents", "start", "end", "series", "cadence", "tax_character"]

rules = {
    name: runs[name].items("scheduled").set_index("name").reindex(columns=_COMPARE)
    for name in SCENARIOS
}
_index = sorted(set().union(*(frame.index for frame in rules.values())))
rules = {name: frame.reindex(_index) for name, frame in rules.items()}

_base = rules[BASELINE]
_differs = pd.DataFrame(False, index=_index, columns=_COMPARE)
for name in VARIANTS:
    other = rules[name]
    _differs |= (_base != other) & ~(_base.isna() & other.isna())

_rows = [
    {"rule": rule, "field": field,
     **{name: rules[name].at[rule, field] for name in SCENARIOS}}
    for rule in _index if _differs.loc[rule].any()
    for field in _COMPARE if _differs.at[rule, field]
]
differences = pd.DataFrame(_rows)
if differences.empty:
    display(Markdown("> ⚠️ **All scenarios have identical rules.**"))
else:
    display(differences.set_index(["rule", "field"]).fillna("—"))

events = pd.concat(
    [runs[name].items("calendar_event").assign(scenario=name) for name in SCENARIOS],
    ignore_index=True,
)
if events.empty:
    display(Markdown("_No calendar events in any scenario._"))
else:
    events["amount"] = (events["amount"].astype("Float64") / 100).map(
        lambda v: "" if pd.isna(v) else f"${v:,.2f}"
    )
    display(
        events[["scenario", "date", "kind", "scheduled_name", "amount", "description"]]
        .set_index(["scenario", "date"])
    )


## Net worth over time

Nominal dollars. (`net_worth_constant_cpi` is available in each run if you want
the inflation-adjusted view later.)


In [ ]:
def _net_worth(name):
    df = runs[name].items("net_worth")[["month", "net_worth"]].copy()
    df["month"] = pd.to_datetime(df["month"])
    return df.set_index("month")["net_worth"] / 100


nw = pd.DataFrame({name: _net_worth(name) for name in SCENARIOS})
delta = nw[VARIANTS].sub(nw[BASELINE], axis=0)

fig, (ax_nw, ax_delta) = plt.subplots(
    2, 1, figsize=(11, 9), sharex=True, height_ratios=[2, 1]
)
for name in SCENARIOS:
    ax_nw.plot(
        nw.index, nw[name], label=name, color=COLORS[name],
        linestyle="-" if name == BASELINE else "--",
        linewidth=2 if name == BASELINE else 1.5,
    )
ax_nw.set_ylabel("net worth (USD)")
ax_nw.set_title("Net worth, nominal")
ax_nw.legend()

for name in VARIANTS:
    ax_delta.plot(delta.index, delta[name], label=name, color=COLORS[name])
ax_delta.axhline(0, color="grey", linewidth=0.8)
ax_delta.set_ylabel(f"difference vs {BASELINE}")
ax_delta.set_xlabel("month")
ax_delta.legend()

for ax in (ax_nw, ax_delta):
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
    ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## Break-even

**Break-even** is the first month after which the variant stays ahead for the rest
of the scenario — not merely the first month it pokes above zero, which can happen
transiently when a lumpy payment lands in one scenario before the other.


In [ ]:
def _breakeven(series):
    """First month after which the variant never falls behind again."""
    ahead = series > 0
    if ahead.all():
        return series.index[0]
    if not ahead.any():
        return None
    after_last_deficit = series.index[series.index > series[~ahead].index.max()]
    return after_last_deficit.min() if len(after_last_deficit) else None


_rows = {}
for name in VARIANTS:
    series = delta[name]
    month = _breakeven(series)
    _rows[name] = {
        "break-even": f"{month:%Y-%m}" if month is not None else "never",
        "deepest deficit": series.min(),
        "when": f"{series.idxmin():%Y-%m}",
        "at scenario end": series.iloc[-1],
        "final net worth": nw[name].iloc[-1],
    }

summary = pd.DataFrame(_rows).T
summary.index.name = f"vs {BASELINE}"
display(
    summary.style.format({
        "deepest deficit": "${:,.0f}",
        "at scenario end": "${:,.0f}",
        "final net worth": "${:,.0f}",
    })
)
display(Markdown(
    f"`{BASELINE}` final net worth: **${nw[BASELINE].iloc[-1]:,.0f}**"
))


## Horizon sensitivity — how much does this matter, and by when?

A claim-timing decision only pays off if you live long enough to collect on it, so
the honest question isn't "what's the difference at 2060" but "what's the
difference **if we stop looking at year X**". Every point on these curves answers
that: the gap at year X *is* the impact of the decision if the money stops
mattering then.

The two panels are deliberately on different scales, because they answer different
questions:

- **The decision** — what waiting is worth. Small, slow, and it starts negative.
- **The risk** — what a Trust Fund shortfall costs. Large, and unrelated to the
  claim date.

If the decision is worth little at your realistic horizons, then taking the
benefit earlier for certainty and convenience costs you very little.


In [ ]:
# The pairs worth contrasting: (scenario, its counterfactual). Edit for other questions.
COMPARISONS = {
    "the decision": {
        "wait — no cut": ("january", "current"),
        "wait — with cut": ("january-cut", "current-cut"),
    },
    "the risk": {
        "cut — claiming now": ("current-cut", "current"),
        "cut — waiting": ("january-cut", "january"),
    },
}
COMPARISONS = {
    group: {label: pair for label, pair in pairs.items()
            if set(pair) <= set(SCENARIOS)}
    for group, pairs in COMPARISONS.items()
}

compare = pd.DataFrame({
    label: nw[a] - nw[b]
    for pairs in COMPARISONS.values() for label, (a, b) in pairs.items()
})

fig, axes = plt.subplots(1, len(COMPARISONS), figsize=(13, 5))
for ax, (group, pairs) in zip(axes, COMPARISONS.items()):
    for offset, label in enumerate(pairs):
        line, = ax.plot(compare.index, compare[label], label=label)
        month = _breakeven(compare[label])
        if month is not None and compare[label].min() < 0:
            ax.axvline(month, color=line.get_color(), linestyle=":", alpha=0.7)
            ax.annotate(
                f"break-even {month:%Y-%m}", xy=(month, 0),
                xytext=(6, 10 + 16 * offset),  # stagger so labels don't collide
                textcoords="offset points", color=line.get_color(), fontsize=9,
            )
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_title(group)
    ax.set_xlabel("if the money stops mattering here…")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
    ax.grid(True, alpha=0.3)
    ax.legend()
axes[0].set_ylabel("difference in net worth")
plt.tight_layout()
plt.show()


In [ ]:
# Year-end value of each comparison, and the pot it is a share of.
horizon = compare.groupby(compare.index.year).tail(1)
horizon.index = horizon.index.year
horizon.index.name = "if we look out to"
_pot = nw[BASELINE].groupby(nw.index.year).tail(1)
_pot.index = _pot.index.year


def _horizon_table(years, frame):
    table = frame.loc[years].copy()
    for label in compare.columns:
        table[f"{label} (% of pot)"] = 100 * frame.loc[years, label] / _pot.loc[years]
    return table.style.format(
        {c: ("{:+.2f}%" if c.endswith("(% of pot)") else "${:+,.0f}")
         for c in table.columns}
    )


### Settling the tax bill at the horizon

Net worth above is after-tax — the engine pays tax as a real transfer — but a tax
year settles the *following* April. So at the end of year X each scenario is
carrying an unpaid liability (or an unclaimed refund) for year X, and net worth
flatters whichever scenario owes more.

Stopping at year X in real life means that bill still has to be paid, so the
comparison below deducts each scenario's outstanding balance. It only moves the
answer where the two scenarios owe *different* amounts — which is exactly the 2026
wage year that motivated the question.


In [ ]:
_SETTLE = ["federal_tax_due", "state_tax_due"]


def _outstanding(name):
    """Year X's liability still unpaid at year end; it settles the April after.

    Only `*_tax_due` is used. `*_tax_overpaid` looks like a refund but also fires
    when the *previous* April's settlement lands in a zero-liability year, so
    crediting it here would double-count that payment.
    """
    df = runs[name].items("tax_year_summary")
    df = df[df["bucket"].isin(_SETTLE)]
    wide = (df.pivot(index="year", columns="bucket", values="amount")
              .reindex(columns=_SETTLE).fillna(0) / 100)
    return wide.sum(axis=1)


outstanding = pd.DataFrame({name: _outstanding(name) for name in SCENARIOS}).fillna(0)

nw_year_end = nw.groupby(nw.index.year).tail(1)
nw_year_end.index = nw_year_end.index.year
settled = nw_year_end.sub(outstanding.reindex(nw_year_end.index).fillna(0), axis=0)

compare_settled = pd.DataFrame({
    label: settled[a] - settled[b]
    for pairs in COMPARISONS.values() for label, (a, b) in pairs.items()
})
compare_settled.index.name = "if we look out to"

_shift = (compare_settled - horizon).abs().max(axis=1)
_moved = _shift[_shift >= 1].index
display(Markdown(
    f"Settling changes the answer in **{len(_moved)} of {len(horizon)}** years"
    + (f" ({', '.join(str(y) for y in _moved)}):" if len(_moved) else ".")
))
if len(_moved):
    display(
        pd.concat(
            {"as-of": horizon.loc[_moved], "settled": compare_settled.loc[_moved]},
            axis=1,
        ).style.format("${:+,.0f}")
    )


In [ ]:
import ipywidgets as widgets

_marks = list(range(horizon.index.min() + 4, horizon.index.max() + 1, 5))
display(Markdown("**At five-year horizons** (tax owed settled):"))
display(_horizon_table(_marks, compare_settled))


def _at_year(year, settle_tax=True):
    """Net worth already reflects tax paid; `settle_tax` also clears what is owed."""
    frame = compare_settled if settle_tax else horizon
    basis = "after settling the year's tax bill" if settle_tax else "as of Dec 31"
    lines = [
        f"**If the money stops mattering at the end of {year}** "
        f"({basis}; net worth {'$'}{_pot.loc[year]:,.0f}):",
        "",
    ]
    lines += [
        f"- {label}: **${value:+,.0f}** ({100 * value / _pot.loc[year]:+.2f}% of the pot)"
        for label, value in frame.loc[year].items()
    ]
    display(Markdown("\n".join(lines)))


widgets.interact(
    _at_year,
    year=widgets.IntSlider(
        min=int(horizon.index.min()), max=int(horizon.index.max()),
        value=2036, description="horizon", continuous_update=False,
    ),
    settle_tax=widgets.Checkbox(value=True, description="settle tax owed"),
);


## Year-end detail


In [ ]:
year_end = nw.groupby(nw.index.year).tail(1)
year_end.index = year_end.index.year
year_end.index.name = "year"
for name in VARIANTS:
    year_end[f"Δ {name}"] = year_end[name] - year_end[BASELINE]
year_end.style.format("${:,.0f}")


## What drives it: Social Security received

Gross benefit (including the Medicare premiums withheld before deposit), so the
two scenarios are compared on the same basis. The cumulative view is the classic
claim-timing picture: the later claim starts lower and crosses over.


In [ ]:
def _ss_gross(name):
    te = runs[name].items("transfer_event")
    te = te[te["tax_character"] == "ss_benefit"].copy()
    te["month"] = pd.to_datetime(te["month"])
    return te.groupby("month")["amount"].sum() / 100


ss = pd.DataFrame({name: _ss_gross(name) for name in SCENARIOS}).fillna(0).sort_index()
ss_cumulative = ss.cumsum()

fig, (ax_month, ax_cum) = plt.subplots(1, 2, figsize=(13, 4))
for name in SCENARIOS:
    ax_month.plot(ss.index, ss[name], label=name, color=COLORS[name], alpha=0.85)
    ax_cum.plot(ss_cumulative.index, ss_cumulative[name], label=name, color=COLORS[name])
ax_month.set_title("Monthly gross Social Security")
ax_cum.set_title("Cumulative gross Social Security")
for ax in (ax_month, ax_cum):
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

lifetime_ss = ss_cumulative.iloc[-1]
display(
    pd.DataFrame({
        "lifetime gross SS": lifetime_ss,
        f"vs {BASELINE}": lifetime_ss - lifetime_ss[BASELINE],
    }).style.format("${:,.0f}")
)


## Tax impact

Claiming earlier stacks Social Security on top of whatever else is taxable that
year. In a year with wages, that is expensive twice over: the benefit itself is
taxable *and* the wages push provisional income past the thresholds, so a larger
fraction of the benefit gets taxed (up to 85%).

Liability is attributed to the **tax year it was earned in**; the cash settles the
following April, so a difference shown for year *Y* leaves the bank account in
*Y+1*. Net worth already reflects this — the table below is the explanation, not
an additional effect.


In [ ]:
_BUCKETS = {
    "federal_tax_liability": "federal tax",
    "state_tax_liability": "state tax",
    "federal_ss_income": "SS gross",
    "federal_taxable_ss": "SS taxed",
}


def _tax(name):
    df = runs[name].items("tax_year_summary")
    df = df[df["bucket"].isin(_BUCKETS)]
    wide = df.pivot(index="year", columns="bucket", values="amount") / 100
    wide = wide.rename(columns=_BUCKETS).reindex(columns=list(_BUCKETS.values()))
    wide["total tax"] = wide["federal tax"] + wide["state tax"]
    return wide


tax = {name: _tax(name) for name in SCENARIOS}
tax_delta = pd.DataFrame({
    name: tax[name]["total tax"] - tax[BASELINE]["total tax"] for name in VARIANTS
}).fillna(0)

# Only the years where somebody's tax bill actually moves are worth reading.
_moved = tax_delta[tax_delta.abs().max(axis=1) >= 1]
display(Markdown(
    f"Tax owed vs `{BASELINE}`, for the {len(_moved)} of {len(tax_delta)} years "
    "where it differs (negative = owes less):"
))
display(_moved.style.format("${:,.0f}").background_gradient(cmap="RdYlGn_r", axis=None))

lifetime_tax = pd.Series({name: tax[name]["total tax"].sum() for name in SCENARIOS})
display(
    pd.DataFrame({
        "lifetime tax": lifetime_tax,
        f"vs {BASELINE}": lifetime_tax - lifetime_tax[BASELINE],
        "lifetime gross SS": lifetime_ss,
        "SS net of tax, vs " + BASELINE: (lifetime_ss - lifetime_ss[BASELINE])
        - (lifetime_tax - lifetime_tax[BASELINE]),
        "net worth at end": nw.iloc[-1],
    }).style.format("${:,.0f}")
)
